# Complexity-03g — Les trois barrières : pourquoi P ≠ NP résiste, et ce qu'une borne de circuit peut encore dire

**Navigation** : [Index](README.md) | [Version Python](Complexity-03g-Trois-Barrieres-Python.ipynb) | [03 — Plus de temps, plus de problèmes](Complexity-03-HierarchieTemporelle-Python.ipynb)

**Objectifs** : distinguer les trois régimes de preuve (relativisation, natural proofs, algébrisation) ; mesurer les deux premières colonnes du tableau de Razborov–Rudich sur une population **complète** ; montrer que le degré d'une extension multilinéaire dépend du corps et ne borne pas la taille des circuits ; reconstruire le plan de Fano et son graphe d'incidence. **Prérequis** : algèbre booléenne, transformée de Möbius, graphes. **Durée estimée** : 90 minutes.

Ce carnet est écrit **depuis une archive datée** du site `pequalnp.com`
(« P vs NP, attacked live by a fleet of AI agents », édité par Brilliant) :
`G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\Complexity Theory and Hardness\2026 - Brilliant - pequalnp.com - P vs NP attacked live by a fleet of AI agents (releve date du 25-11-2026)\`
(manifeste de provenance dans le même dossier : URL, heure UTC, sha256, taille).

**Ce que le jumeau change, et pourquoi.** Les calculs sont **BCL purs** : pas de
NumPy, pas de NuGet. Ce qui change entre les deux langages est nommé à chaque
fois : les **chronomètres** (le JIT n'a pas les mêmes coûts que NumPy) et le
**flux aléatoire** du jouet d'oracles (`System.Random` n'est pas le MT19937 de
NumPy). Ce qui **ne** change pas est nommé aussi : la famille pseudo-aléatoire
dérivée de `SHA-256` est **reproductible à l'identique** dans les deux langages,
et la dernière cellule **vérifie** par assertion que les valeurs mesurées ici
sont celles du jumeau Python.


## 0. Pourquoi ce carnet

La série Complexity traitait la relativisation et le mur d'énumération ; elle ne
nommait ni les **natural proofs**, ni l'**algébrisation** (mesuré : 0 occurrence de
`natural proof`, `Razborov`, `Rudich`, `algebrization` dans la série avant ce carnet).
Ces deux barrières ne sont pas des variantes de la première : chacune restreint
d'une façon différente **ce qu'un argument de borne inférieure a le droit de faire**.

En 2026, une flotte d'agents a attacked P vs NP publiquement, avec un juge Lean
unique et un registre d'énoncés. Ce dispositif est un **cas d'école d'épistémologie
d'une annonce en cours** : ce qu'on peut y citer, ce qu'on ne peut pas y re-prouver,
et ce qui arrive à un énoncé quand il est réécrit.

Prérequis ajoutés par ce carnet : la notion de **table de vérité** et de
**transformée de Möbius** sur GF(2) (rappelée en §2), la **matrice d'incidence**
d'un plan projectif (rappelée en §4). Rien d'autre n'est supposé.


In [1]:
// --- Parametres -------------------------------------------------------------
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
using System.Numerics;
using System.Security.Cryptography;
using System.Text;
using System.Text.RegularExpressions;
using System.Diagnostics;
using System.IO;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
const int SEED = 20261125;
var rng = new Random(SEED);

Console.WriteLine($".NET {Environment.Version} ; graine {SEED}");


The below script needs to be able to find the current output cell; this is an easy method to get it.

.NET 10.0.15 ; graine 20261125


## 1. Les trois barrières, ce qu'elles interdisent

| Barrière | Énoncé **attribué** | Ce qu'elle interdit | Statut ici |
|---|---|---|---|
| Relativisation (Baker–Gill–Solovay, 1975) | Il existe un oracle `A` tel que `P^A = NP^A` et un oracle `B` tel que `P^B ≠ NP^B`. | Toute preuve qui ne tient pas dans **les deux** mondes. | traité en [03](Complexity-03-HierarchieTemporelle-Python.ipynb) et [03c](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) — **rappelé, pas re-démontré** |
| Natural proofs (Razborov–Rudich, STOC '94 ; JCSS 55(1), 1997) | Sous l'hypothèse de fonctions à sens unique fortes, aucun *natural proof* (grande, constructive, utile) ne prouve `P ≠ NP`. | Une borne inférieure « naturelle » (densité ≥ 1/2, calculable en temps polynomial depuis la table de vérité) **utile** pour une borne de circuits. | **mesuré** en §2 : deux premières colonnes, la troisième est le mur |
| Algébrisation (Aaronson–Wigderson, STOC 2008 ; version étendue TOCT 1(1), 2009) | Si `P ≠ NP` alors `P ≠ NP` **relativisé à un oracle algébrique** (l'oracle calcule des polynômes multilinéaires sur GF(p)). | Une preuve qui survit à la relativisation mais **pas** à l'algébrisation. | **mesuré** en §3 : degrés d'interpolation, puis un jouet d'oracles |

**Trois régimes, pas trois variantes.** La relativisation dit « la preuve doit
tenir dans deux mondes » ; les natural proofs disent « la preuve doit être
naturelle *et* ne peut pas l'être » ; l'algébrisation ajoute un troisième monde
où l'oracle est **calculable par un polynôme**.


In [2]:
// Les trois barrieres : un en-tete execute, pour que le tableau ci-dessus ne
// soit qu'une projection d'une structure unique (nom, annee, source, regime).
record Barriere(string Nom, string Auteurs, int Annee, string Source, string Regime, string TraiteeAilleurs);

var barrieres = new List<Barriere> {
    new("relativisation", "Baker, Gill, Solovay", 1975, "SIAM J. Comput. 4(4)", "deux mondes", "03 et 03c de cette serie"),
    new("natural proofs", "Razborov, Rudich", 1994, "STOC '94 ; J. Comput. Syst. Sci. 55(1) 1997", "le monde naturel est mur", "mesure ici (section 2)"),
    new("algebrization", "Aaronson, Wigderson", 2008, "STOC 2008 ; version etendue TOCT 1(1) 2009", "un troisieme monde polynomial", "mesure ici (section 3)"),
};

foreach (var b in barrieres)
    Console.WriteLine($"{b.Nom,18} ({b.Annee})  regime : {b.Regime,-28}  -> {b.TraiteeAilleurs}");

Console.WriteLine();
Console.WriteLine("Registre natif des trois sources : " + string.Join("; ", barrieres.Select(b => b.Source)));


    relativisation (1975)  regime : deux mondes                   -> 03 et 03c de cette serie


    natural proofs (1994)  regime : le monde naturel est mur      -> mesure ici (section 2)


     algebrization (2008)  regime : un troisieme monde polynomial  -> mesure ici (section 3)


Registre natif des trois sources : SIAM J. Comput. 4(4); STOC '94 ; J. Comput. Syst. Sci. 55(1) 1997; STOC 2008 ; version etendue TOCT 1(1) 2009


## 2. Natural proofs, mesurés sur des tables de vérité

Une **propriété** d'une famille de fonctions booléennes est dite :

- **grande** (*large*) si elle vaut `vrai` pour au moins une fraction `1/2` des
  fonctions ;
- **constructive** si, donnée la table de vérité, on peut décider la propriété en
  temps polynomial en `n` ;
- **utile** (*useful*) si elle ne se réduit pas à une tautologie et peut servir à
  établir une borne inférieure de circuits.

Les trois conditions sont **attribuées** à Razborov–Rudich (1994) ; le théorème
est **cité, non re-démontré** ici. Ce que le carnet mesure, ce sont les deux
premières colonnes, et il **nomme** la troisième comme le mur.

### Témoin 1 — la densité et le coût, colonne par colonne

Sur **toutes** les fonctions booléennes à `n = 4` variables — la population
complète, `2^16 = 65 536` fonctions de 16 entrées — on mesure : le **degré
algébrique** (transformée de Möbius sur GF(2)), la **sensibilité moyenne** (on
retourne chaque entrée), et la parité. Les densités sont donc **exactes**, pas
estimées.


In [3]:
// --- Témoin 1 : la population complète et les deux grandeurs ----------------
// Une table de verite est un entier dont le bit x vaut f(x) : pour n = 4, ce
// sont 16 bits, et la POPULATION COMPLETE des 2^16 = 65 536 fonctions est
// exactement l'ensemble des entiers 0..65 535 (une fonction = les bits de son
// propre numero).
int[] ToutesFonctions(int n) => Enumerable.Range(0, 1 << (1 << n)).ToArray();

/// Transformee de Moebius sur GF(2), appliquee a une table : le « papillon »
/// de l'ANF (c[S] = XOR des f(T) pour T sous-ensemble de S).
int Moebius(int table, int n) {
    int t = table;
    for (int i = 0; i < n; i++) {
        int step = 1 << i;
        for (int bloc = 0; bloc < (1 << n); bloc += 2 * step)
            for (int j = 0; j < step; j++) {
                int lo = bloc + j, hi = lo + step;
                int bLo = (t >> lo) & 1, bHi = (t >> hi) & 1;
                if ((bLo ^ bHi) == 1) t |= 1 << hi; else t &= ~(1 << hi);
            }
    }
    return t;
}

/// Degres algebriques (ANF sur GF(2)) de chaque table de verite : le poids d'un
/// monome est le nombre de variables qu'il contient, et le degre est le plus
/// grand poids present dans l'ANF.
int[] DegresAlgebriques(int[] tables, int n) {
    var degres = new int[tables.Length];
    for (int k = 0; k < tables.Length; k++) {
        int anf = Moebius(tables[k], n);
        int deg = 0;
        for (int s = 0; s < (1 << n); s++)
            if (((anf >> s) & 1) == 1) deg = Math.Max(deg, BitOperations.PopCount((uint)s));
        degres[k] = deg;
    }
    return degres;
}

/// Sensibilite moyenne : fraction des entrees dont le retournement change la
/// sortie. On compare f(x) et f(x xor 2^i) -- la VRAIE voisine -- et non pas la
/// sortie retournee ; chaque arete du cube est rencontree deux fois, d'ou la
/// division par 2*n.
double[] SensibilitesMoyennes(int[] tables, int n) {
    var sens = new double[tables.Length];
    for (int k = 0; k < tables.Length; k++) {
        int t = tables[k], cumule = 0;
        for (int i = 0; i < n; i++) {
            int voisin = 1 << i;
            for (int x = 0; x < (1 << n); x++)
                if (((t >> x) & 1) != ((t >> (x ^ voisin)) & 1)) cumule++;
        }
        sens[k] = cumule / (double)(2 * n);
    }
    return sens;
}

int n = 4;
var tables = ToutesFonctions(n);
Console.WriteLine($"n = {n} variables -> population complete : {tables.Length} fonctions de {1 << n} entrees");

var degres = DegresAlgebriques(tables, n);
var sensib = SensibilitesMoyennes(tables, n);

Console.WriteLine();
Console.WriteLine($"degre maximal : {degres.Max()}");
Console.WriteLine($"fonctions de degre n (la parite et ses proches) : {degres.Count(d => d == n)}");
Console.WriteLine($"densite parite (deg == n) : {degres.Count(d => d == n) / (double)tables.Length}");


n = 4 variables -> population complete : 65536 fonctions de 16 entrees


degre maximal : 4


fonctions de degre n (la parite et ses proches) : 32768


densite parite (deg == n) : 0,5


### Lecture du résultat — deux colonnes mesurées, la troisième nommée

La parité a un **degré algébrique maximal** et une **densité exactement 1/2** :
elle est grande au sens de Razborov–Rudich. Elle est aussi **constructive** : le
degré se calcule depuis la seule table de vérité, en `O(n · 2^n)` opérations,
sans rien supposer de la structure du circuit.

Mais la parité est **inutile** pour une borne de circuits : elle est vraie
exactement pour une fonction sur deux, et son complémentaire est faux pour une
fonction sur deux. C'est là que se place le mur du théorème : une propriété
naturelle doit être **grande** *et* **constructive** *et* **utile**, et c'est la
troisième condition que le théorème rend inatteignable sous l'hypothèse de
fonctions à sens unique fortes.

Le tableau ci-dessous mesure les deux premières colonnes sur les mêmes 65 536
fonctions ; la troisième est **hors de portée**, et le carnet le dit.


In [4]:
// --- Temoin 1 bis : le tableau des proprietes, densite + cout de calcul -----
/// Densites des proprietes mesurees, sur un lot de tables de verite.
Dictionary<string, double> Densites(int[] lot, int[] degresLot, double[] sensibLot, int n) {
    var dens = new Dictionary<string, double>();
    dens["parite (deg == n)"] = degresLot.Count(d => d == n) / (double)lot.Length;
    for (int d = 1; d <= n; d++) dens[$"deg >= {d}"] = degresLot.Count(x => x >= d) / (double)lot.Length;
    foreach (int s in new[] { 2, 4, 6 }) dens[$"sensibilite moyenne >= {s}"] = sensibLot.Count(x => x >= s) / (double)lot.Length;
    return dens;
}

/// Famille pseudo-aleatoire : premier bit de SHA-256(cle || x).
/// Sert de temoin negatif : les tests « naturels » ne doivent pas la
/// distinguer d'une famille tiree au hasard. La cle et les charges dependent
/// seulement de la graine et de SHA-256 : la famille est donc REPRODUISIBLE A
/// L'IDENTIQUE dans le jumeau Python.
int[] PseudoFamille(int n, int k) {
    var lot = new int[k];
    for (int j = 0; j < k; j++) {
        byte[] cle = SHA256.HashData(Encoding.UTF8.GetBytes($"{SEED}-{j}"));
        int t = 0;
        for (int x = 0; x < (1 << n); x++) {
            byte[] charge = new byte[cle.Length + 2];
            Buffer.BlockCopy(cle, 0, charge, 0, cle.Length);
            charge[cle.Length] = (byte)(x & 0xFF);
            charge[cle.Length + 1] = (byte)((x >> 8) & 0xFF);
            if ((SHA256.HashData(charge)[0] & 1) == 1) t |= 1 << x;
        }
        lot[j] = t;
    }
    return lot;
}

var mesurees = Densites(tables, degres, sensib, n);

// Cout de calcul : le prix pour DECIDER la propriete depuis la table de verite.
var chrono = Stopwatch.StartNew();
for (int rep = 0; rep < 20; rep++) { var _ = DegresAlgebriques(tables, n); }
double tDegree = chrono.Elapsed.TotalMilliseconds / 20;
chrono.Restart();
for (int rep = 0; rep < 20; rep++) { var __ = SensibilitesMoyennes(tables, n); }
double tSensib = chrono.Elapsed.TotalMilliseconds / 20;

Console.WriteLine($"{"propriete",-34} {"densite",9}  {"constructive ?",-14} utile ?");
Console.WriteLine(new string('-', 78));
foreach (var kv in mesurees)
    Console.WriteLine($"{kv.Key,-34} {kv.Value,9:F4}  {"oui",-14} {(kv.Key == "parite (deg == n)" ? "hors de portee" : "?")}");

Console.WriteLine();
Console.WriteLine($"cout decideur de degre  : {tDegree,8:F2} ms pour {tables.Length} tables ({n} variables)");
Console.WriteLine($"cout decideur de sensib.: {tSensib,8:F2} ms pour {tables.Length} tables ({n} variables)");

// --- Temoin negatif : la famille pseudo-aleatoire ---------------------------
int K = 256;
var famille = PseudoFamille(n, K);
var degrPseudo = DegresAlgebriques(famille, n);
var sensPseudo = SensibilitesMoyennes(famille, n);
var mesPseudo = Densites(famille, degrPseudo, sensPseudo, n);

Console.WriteLine();
Console.WriteLine($"Comparaison population complete ({tables.Length} fonctions) / famille pseudo-aleatoire (K = {K}) :");
var Proprietes = new[] { "parite (deg == n)", "deg >= 3", "sensibilite moyenne >= 6" };
var sigmaObs = new Dictionary<string, double>();
foreach (var cle in Proprietes) {
    double pPop = mesurees[cle], pPseudo = mesPseudo[cle];
    double et = Math.Sqrt(pPop * (1 - pPop) / K);
    double sigma = (pPseudo - pPop) / et;
    sigmaObs[cle] = sigma;
    string sigmaTxt = (sigma >= 0 ? "+" : "-") + Math.Abs(sigma).ToString("F2");
    Console.WriteLine($"  {cle,-30} population = {pPop:F4} | pseudo = {pPseudo:F4} | ecart = {sigmaTxt} sigma");
}

// --- Reference : la loi des memes ecarts pour un tirage ALEATOIRE ---------
// Constater trois ecarts ne dit rien tant qu'on ne sait pas a quoi les
// comparer. On tire R fois K fonctions au hasard dans la population
// complete, et on mesure la loi des memes ecarts. Le PRNG est SplitMix64,
// dont les 64 bits se reproduisent a l'identique dans le jumeau Python :
// la parite porte donc aussi sur cette mesure.
sealed class SplitMix64 {
    ulong etat;
    public SplitMix64(ulong graine) { etat = graine; }
    public ulong Suivant() {
        etat += 0x9E3779B97F4A7C15UL;
        ulong z = etat;
        z = (z ^ (z >> 30)) * 0xBF58476D1CE4E5B9UL;
        z = (z ^ (z >> 27)) * 0x94D049BB133111EBUL;
        return z ^ (z >> 31);
    }
}

int R = 200;
var prng = new SplitMix64((ulong)SEED);
var ecartsRef = new Dictionary<string, List<double>>();
var ecartTypeRef = new Dictionary<string, double>();
var partRef = new Dictionary<string, double>();
foreach (var cle in Proprietes) ecartsRef[cle] = new List<double>();
for (int rep = 0; rep < R; rep++) {
    var tirage = new int[K];
    // 65 536 = 2**16 divise 2**64 : le modulo est donc sans biais.
    for (int j = 0; j < K; j++) tirage[j] = (int)(prng.Suivant() % (ulong)tables.Length);
    var degrTirage = DegresAlgebriques(tirage, n);
    var sensTirage = SensibilitesMoyennes(tirage, n);
    var mesTirage = Densites(tirage, degrTirage, sensTirage, n);
    foreach (var cle in Proprietes) {
        double pPop = mesurees[cle];
        double et = Math.Sqrt(pPop * (1 - pPop) / K);
        ecartsRef[cle].Add((mesTirage[cle] - pPop) / et);
    }
}
Console.WriteLine();
Console.WriteLine($"Reference : {R} tirages aleatoires de K = {K} fonctions dans la population complete");
Console.WriteLine($"{"propriete",-30} {"ecart-type",12} {"part |ecart| >= observe",24} {"observe",12}");
Console.WriteLine(new string('-', 82));
foreach (var cle in Proprietes) {
    var vals = ecartsRef[cle];
    double moy = vals.Average();
    double ecartType = Math.Sqrt(vals.Sum(v => (v - moy) * (v - moy)) / (vals.Count - 1));
    double part = (double)vals.Count(v => Math.Abs(v) >= Math.Abs(sigmaObs[cle])) / vals.Count;
    ecartTypeRef[cle] = ecartType;
    partRef[cle] = part;
    Console.WriteLine($"{cle,-30} {ecartType,12:F2} {(100 * part).ToString("F1"),23} % {(sigmaObs[cle] >= 0 ? "+" : "-") + Math.Abs(sigmaObs[cle]).ToString("F2")} sigma");
}


propriete                            densite  constructive ? utile ?


------------------------------------------------------------------------------


parite (deg == n)                     0,5000  oui            hors de portee


deg >= 1                              1,0000  oui            ?


deg >= 2                              0,9995  oui            ?


deg >= 3                              0,9688  oui            ?


... (12 outputs left out)

----------------------------------------------------------------------------------


parite (deg == n)                      1,03                    58,0 % +0,62 sigma


deg >= 3                               1,01                    38,0 % +1,08 sigma


sensibilite moyenne >= 6               1,02                    48,0 % -0,73 sigma


### Lecture du résultat — trois écarts de l'ordre de l'unité, comme un tirage aléatoire

Les mêmes propriétés, mesurées sur une famille tirée de `SHA-256` (K = 256
échantillons), donnent des densités **comparables** à celles de la population :
les écarts valent **+0,62 σ**, **+1,08 σ** et **−0,73 σ**, en unités de
l'écart-type d'un tirage de 256 fonctions.

Constater trois écarts ne suffirait pas à dire que la famille ne se distingue
pas : il faudrait la loi contre laquelle ils se lisent. Le carnet la **mesure** —
il tire **200 fois** 256 fonctions au hasard dans la population complète et
regarde la loi des mêmes écarts. Cette loi a un écart-type de **1,03**, **1,01**
et **1,02**, et les trois écarts observés sont atteints ou dépassés par
**58,0 %**, **38,0 %** et **48,0 %** des tirages aléatoires : ils sont
**ordinaires**. Aucun des trois ne sort de ce qu'un tirage au hasard produit
couramment.

Les tests « naturels » acceptent donc cette famille comme ils accepteraient du
hasard. C'est exactement le point du troisième régime, et c'est pourquoi la
troisième colonne du tableau reste **hors de portée** : trouver une propriété
grande, constructive **et** utile serait précisément construire ce que le
théorème attribué rend impossible sous hypothèse forte.

**Ce que ce carnet ne fait pas** : il ne prouve pas le théorème de
Razborov–Rudich, il ne l'attribue pas non plus comme une conséquence de la
mesure ci-dessus. La mesure porte sur les **deux premières colonnes** ;
l'hypothèse (existence de fonctions à sens unique fortes) est **attribuée** et
non vérifiée ici.


## 3. Algébrisation : le degré d'un polynôme multilinéaire

L'extension multilinéaire d'une fonction `f : {0,1}^n → F_p` sur un corps `F_p`
est l'unique polynôme multilinéaire `f̃` tel que `f̃(1_T) = f(T)` pour tout
sous-ensemble `T ⊆ [n]`. Dans la base des « boîtes », ses coefficients s'obtiennent
par inversion de Möbius :

```
c_S = Σ_{T ⊆ S} (-1)^{|S|-|T|} · f(1_T)     (mod p)
```

Sur GF(2), cette formule redonne l'ANF de la section 2 ; sur un corps quelconque,
les coefficients sont des différences. Le **degré** de `f̃` est le plus grand `|S|`
tel que `c_S ≠ 0`.

Attention : ce degré **dépend du corps**. Sur GF(2) — l'ANF usuelle — le degré est
un invariant de la fonction booléenne ; sur GF(p) avec `p` impair, l'extension
multilinéaire d'une fonction **booléenne** (à valeurs dans `{0,1}`) peut avoir un
tout autre degré. C'est mesuré ci-dessous, et c'est un fait important : le « degré
algébrique » n'est pas un invariant de la fonction, mais un invariant de la
fonction **vue dans un corps**.

Le tableau mesure donc deux choses : le degré (sur GF(2), GF(3), GF(5)) et le
**nombre de portes** d'un circuit explicite en portes AND/OR/NOT — un circuit
**construit** puis **compté**, jamais une porte affirmée.


In [5]:
// --- Algebrization : degre d'extension et nombre de portes -----------------
/// Inversion de Moebius sur Z/p : c[S] = somme_{T sous-ensemble de S} de
/// (-1)^{|S|-|T|} f(1_T), calculee en place (la moitie haute moins la basse).
int[] MoebiusModP(int[] f, int p) {
    int n = BitOperations.Log2((uint)f.Length);
    var a = (int[])f.Clone();
    for (int i = 0; i < n; i++) {
        int step = 1 << i;
        for (int bloc = 0; bloc < a.Length; bloc += 2 * step)
            for (int j = 0; j < step; j++) {
                int lo = bloc + j, hi = lo + step;
                a[hi] = ((a[hi] - a[lo]) % p + p) % p;
            }
    }
    return a;
}

/// Degré de l'extension multilineaire sur GF(p) d'une table de verite.
int DegreeExtension(int[] f, int p) {
    var c = MoebiusModP(f, p);
    int deg = 0;
    for (int s = 0; s < c.Length; s++)
        if (c[s] != 0) deg = Math.Max(deg, BitOperations.PopCount((uint)s));
    return deg;
}

/// AND, OR, PARITY, MAJORITY sur n variables, en tables de verite.
Dictionary<string, int[]> FonctionsDeReference(int n) {
    int m = 1 << n;
    var andF = new int[m]; var orF = new int[m]; var parF = new int[m]; var majF = new int[m];
    for (int x = 0; x < m; x++) {
        int somme = 0, produit = 1;
        for (int i = 0; i < n; i++) { int b = (x >> i) & 1; somme += b; produit *= b; }
        andF[x] = produit; orF[x] = somme > 0 ? 1 : 0; parF[x] = somme % 2; majF[x] = somme > n / 2 ? 1 : 0;
    }
    return new Dictionary<string, int[]> { ["AND"] = andF, ["OR"] = orF, ["PARITY"] = parF, ["MAJORITY"] = majF };
}

/// Compteur de portes AND/OR/NOT : le circuit est CONSTRUIT, puis compte.
/// Une reference est soit un nom d'entree, soit "g<index>" (sortie de porte).
/// Les operandes sont memorises en plus des natures : c'est ce qui permet
/// d'EVALUER le circuit, donc de controler qu'il calcule la fonction.
class Circuit {
    public List<string> Entrees { get; }
    public List<string> Gates { get; } = new();
    public List<(string a, string b)> Operandes { get; } = new();
    public Circuit(IEnumerable<string> entrees) { Entrees = entrees.ToList(); }
    public string AND(string a, string b) { Gates.Add("AND"); Operandes.Add((a, b)); return "g" + (Gates.Count - 1); }
    public string OR(string a, string b) { Gates.Add("OR"); Operandes.Add((a, b)); return "g" + (Gates.Count - 1); }
    public string NOT(string a) { Gates.Add("NOT"); Operandes.Add((a, null)); return "g" + (Gates.Count - 1); }
    /// XOR en 5 portes : NOT a, NOT b, AND, AND, OR.
    public string XOR(string a, string b) { string na = NOT(a), nb = NOT(b); return OR(AND(a, nb), AND(na, b)); }
    /// Majorite de trois : (a AND b) OR (c AND (a OR b)) -- 4 portes.
    public string MAJ3(string a, string b, string c) { return OR(AND(a, b), AND(c, OR(a, b))); }
    public int NPortes => Gates.Count;
}

int n7 = 7;  // impair : le degre de MAJORITY n'est PAS n -- il est mesure dans le tableau
var refs = FonctionsDeReference(n7);

// Circuits explicites : le nombre de portes est COMPTE, jamais affirme.
// AND : n-1 portes ; OR : n-1 portes ; PARITY : (n-1) XOR de 5 portes.
var cAnd = new Circuit(Enumerable.Range(0, n7).Select(i => $"x{i}"));
string tAnd = "x0";
for (int i = 1; i < n7; i++) tAnd = cAnd.AND(tAnd, $"x{i}");
int nAnd = cAnd.NPortes;

var cOr = new Circuit(Enumerable.Range(0, n7).Select(i => $"x{i}"));
string tOr = "x0";
for (int i = 1; i < n7; i++) tOr = cOr.OR(tOr, $"x{i}");
int nOr = cOr.NPortes;

var cPar = new Circuit(Enumerable.Range(0, n7).Select(i => $"x{i}"));
string tPar = "x0";
for (int i = 1; i < n7; i++) tPar = cPar.XOR(tPar, $"x{i}");
int nPar = cPar.NPortes;

/// Circuit du seuil « au moins k entrees a 1 » : la disjonction des
/// sous-ensembles de taille k dont toutes les entrees sont a 1. Il en
/// existe un exactement quand au moins k entrees sont a 1. C'est une
/// borne haute de construction, pas un optimum : le carnet compte, il
/// n'affirme pas.
(Circuit circuit, string sortie) CircuitSeuil(string[] entrees, int k) {
    var c = new Circuit(entrees);
    var termes = new List<string>();
    for (int masque = 0; masque < (1 << entrees.Length); masque++) {
        if (BitOperations.PopCount((uint)masque) != k) continue;
        string terme = null;
        for (int i = 0; i < entrees.Length; i++) {
            if ((masque & (1 << i)) == 0) continue;
            terme = terme == null ? entrees[i] : c.AND(terme, entrees[i]);
        }
        termes.Add(terme);
    }
    string sortie = termes[0];
    for (int i = 1; i < termes.Count; i++) sortie = c.OR(sortie, termes[i]);
    return (c, sortie);
}

/// Evalue un circuit sur une affectation des entrees (0 ou 1).
int Evaluer(Circuit c, string sortie, int[] affectation) {
    var env = new Dictionary<string, int>();
    for (int i = 0; i < c.Entrees.Count; i++) env[c.Entrees[i]] = affectation[i];
    for (int i = 0; i < c.Gates.Count; i++) {
        int a = env[c.Operandes[i].a];
        int v = c.Gates[i] switch {
            "AND" => a & env[c.Operandes[i].b],
            "OR" => a | env[c.Operandes[i].b],
            _ => 1 - a,
        };
        env["g" + i] = v;
    }
    return env[sortie];
}

// MAJORITY sur 7 entrees : le seuil « au moins 4 sur 7 », construit par la
// disjonction des 35 quadruplets -- chaque entree du carnet est utilisee,
// et ONLY celle-la.
var (cMaj, sortieMaj) = CircuitSeuil(Enumerable.Range(0, n7).Select(i => $"x{i}").ToArray(), (n7 + 1) / 2);
int nMaj = cMaj.NPortes;

var cAnd2 = new Circuit(new[] { "x0", "x1" });
cAnd2.AND("x0", "x1");
int nAnd2 = cAnd2.NPortes;

Console.WriteLine($"n = {n7} variables (impair) -- degre algebrique contre nombre de portes");
Console.WriteLine($"{"fonction",-22}" + string.Concat(new[] { 2, 3, 5 }.Select(p => $"{"deg p=" + p,11}")) + $"{"portes",9}");
Console.WriteLine(new string('-', 66));
foreach (var (nom, f, nb) in new[] {
    ("AND (n variables)", refs["AND"], nAnd),
    ("OR (n variables)", refs["OR"], nOr),
    ("PARITY (n variables)", refs["PARITY"], nPar),
    ("MAJORITY (n variables)", refs["MAJORITY"], nMaj),
}) {
    var ligne = nom.PadRight(22);
    foreach (int p in new[] { 2, 3, 5 }) ligne += DegreeExtension(f, p).ToString().PadLeft(11);
    Console.WriteLine(ligne + nb.ToString().PadLeft(9));
}
var fAnd2 = new[] { 0, 0, 0, 1 };  // x0 AND x1 sur 2 variables
var ligne2 = "AND_2 (une seule porte)".PadRight(22);
foreach (int p in new[] { 2, 3, 5 }) ligne2 += DegreeExtension(fAnd2, p).ToString().PadLeft(11);
Console.WriteLine(ligne2 + nAnd2.ToString().PadLeft(9));

// Controle : le circuit MAJORITY calcule-t-il vraiment la fonction ? On le
// verifie sur les 2**n affectations -- on ne l'affirme pas.
int nOkMaj = 0;
for (int masque = 0; masque < (1 << n7); masque++) {
    var affectation = Enumerable.Range(0, n7).Select(i => (masque >> i) & 1).ToArray();
    int attendu = affectation.Sum() >= (n7 + 1) / 2 ? 1 : 0;
    if (Evaluer(cMaj, sortieMaj, affectation) == attendu) nOkMaj++;
}
Console.WriteLine($"controle : le circuit MAJORITY calcule la fonction sur {nOkMaj} / {1 << n7} affectations");

// Verification de la construction sur un cas dont la reponse est connue :
// l'indicateur de la fonction z1 z2 z3 a pour ANF z1 z2 z3 (degre 3).
var f = new int[8];
for (int x = 0; x < 8; x++) {
    var z = Enumerable.Range(0, 3).Select(i => (x >> i) & 1).ToArray();
    f[x] = z[0] * z[1] * z[2];
}
Console.WriteLine();
Console.WriteLine("controle : degre de z1 z2 z3 = " + DegreeExtension(f, 2) + " (attendu 3)");


n = 7 variables (impair) -- degre algebrique contre nombre de portes


fonction                  deg p=2    deg p=3    deg p=5   portes


------------------------------------------------------------------


AND (n variables)               7          7          7        6


OR (n variables)                7          7          7        6


PARITY (n variables)            1          7          7       30


MAJORITY (n variables)          4          7          5      139


AND_2 (une seule porte)          2          2          2        1


controle : le circuit MAJORITY calcule la fonction sur 128 / 128 affectations


controle : degre de z1 z2 z3 = 3 (attendu 3)


### Lecture du résultat — deux degrés pour la même fonction booléenne

Trois faits se lisent dans le tableau :

- **sur GF(2)** (l'ANF), `AND_7` et `OR_7` sont de degré 7, `PARITY_7` de degré **1**
  — une fonction *linéaire* ! — et `MAJORITY_7` de degré 4 ;
- **sur GF(3)**, les quatre remontent à 7 : la **même fonction booléenne** a un
  **degré différent** selon le corps ;
- **le nombre de portes ne suit pas le degré** : `AND_2` est **une seule porte**
  et porte un degré 2, quand `PARITY_7` coûte 30 portes pour un degré 1 sur GF(2).
  Le seuil `MAJORITY_7` coûte **139 portes** dans la construction directe du
  carnet — la disjonction des 35 quadruplets — dont le calcul de la fonction
  est **contrôlé sur les 128 affectations** : un seuil coûte bien plus cher
  qu'une conjonction, et ce nombre est un majorant de la construction, pas un
  optimum.

Le troisième point est le plus utile : le degré n'est **pas** une borne inférieure
sur la taille des circuits — `AND_2` le disprose en une ligne. Un invariant calculé
dans un corps ne se transporte donc pas d'un monde à l'autre : c'est exactement la
raison d'être d'un monde où l'oracle calcule des polynômes sur un corps **choisi**.

Le **témoin 2** est cette table. Reste le **jouet d'oracles** : pourquoi la
relativisation ne se transporte-t-elle pas telle quelle au monde algébrique ? On le
mesure sur un cas concret.


In [6]:
// --- Temoin 2 bis : le jouet d'oracles --------------------------------------
// Deux mondes de MEME cardinalite, dans le MEME espace : GF(2)^4 prive de {0},
// 15 points. Monde A : le plan de Fano, transporte (7 points, 7 droites).
// Monde B : 7 points tires au hasard dans les 15. Un test NATUREL (compter les
// points) ne les distingue jamais ; un test ALGEBRIQUE (x + y + z = 0) en donne
// des comptes tres differents.

/// Nombre de triplets {x, y, z} distincts de S tels que x + y + z = 0.
/// Chaque triplet est compte par ses trois paires, d'ou la division.
int TripletsNuls(HashSet<int> S) {
    int paires = 0;
    foreach (int a in S)
        foreach (int b in S)
            if (a < b && S.Contains(a ^ b)) paires++;
    return paires / 3;
}

var fano = Enumerable.Range(1, 7).Select(v => v << 1).ToList();  // plan de Fano dans GF(2)^4
var ambient = Enumerable.Range(1, 15).ToList();                   // GF(2)^4 prive de {0}
int tripletsFano = TripletsNuls(new HashSet<int>(fano));

var comptages = new List<int>();
for (int essai = 0; essai < 2000; essai++) {
    var tirage = ambient.OrderBy(_ => rng.Next()).Take(7).ToHashSet();
    comptages.Add(TripletsNuls(tirage));
}
double moyenne = comptages.Average();
int maximum = comptages.Max();
double partFano = comptages.Count(c => c == tripletsFano) / (double)comptages.Count;

Console.WriteLine($"monde A -- plan de Fano ({fano.Count} points) : {tripletsFano} triplets x + y + z = 0");
Console.WriteLine($"monde B -- 7 points tires au hasard          : moyenne {moyenne:F3}, max {maximum}, part des tirages a {tripletsFano} : {partFano:F4}");

// Test naturel : le nombre de points. Il ne distingue RIEN.
Console.WriteLine();
Console.WriteLine($"test naturel (cardinalite)  : monde A = {fano.Count} points, monde B = 7 points -> indistinguables");

/// Degré algebrique de l'indicateur d'un ensemble de points dans GF(2)^n_bits.
int DegreeIndicateur(IEnumerable<int> ensemble, int nBits) {
    var f = new int[1 << nBits];
    var S = new HashSet<int>(ensemble);
    for (int x = 0; x < (1 << nBits); x++) f[x] = S.Contains(x) ? 1 : 0;
    return DegreeExtension(f, 2);
}

int degA = DegreeIndicateur(fano, 4);
var tirageFinal = ambient.OrderBy(_ => rng.Next()).Take(7).ToList();
int degB = DegreeIndicateur(tirageFinal, 4);
Console.WriteLine($"degre algebrique de l'indicateur : monde A = {degA} | monde B = {degB}");
Console.WriteLine("=> le DEGRE ne distingue pas les deux mondes ; la STRUCTURE algebrique, oui.");


monde A -- plan de Fano (7 points) : 7 triplets x + y + z = 0


monde B -- 7 points tires au hasard          : moyenne 2,687, max 7, part des tirages a 7 : 0,0020


test naturel (cardinalite)  : monde A = 7 points, monde B = 7 points -> indistinguables


degre algebrique de l'indicateur : monde A = 4 | monde B = 4


=> le DEGRE ne distingue pas les deux mondes ; la STRUCTURE algebrique, oui.


### Lecture du résultat — deux mondes de même densité, un seul se distingue

Les deux mondes ont la **même densité** (7 points sur 15) et le **même degré**
algébrique de leur indicateur : aucun test « naturel » ne les sépare, exactement
comme dans la section 2. Un test qui pose la question *algébrique* — le nombre de
triplets `x + y + z = 0` — les sépare presque toujours : le plan de Fano en a 7,
un tirage aléatoire en a **2,7 en moyenne**, et n'atteint 7 que dans **0,20 %**
des tirages (mesuré sur 2 000 tirages ; le jumeau Python en compte **0,35 %**,
avec un autre générateur).

**Ce qui diffère du jumeau Python** : le tirage du monde B. Python utilise
`numpy.random` (MT19937), C# utilise `System.Random` : les **instances** diffèrent,
donc la moyenne et la part exactes diffèrent aussi. Le nombre de triplets du
**monde A** (le plan de Fano) est déterministe et vaut 7 dans les deux carnets.

C'est le sens de l'algébrisation : **un monde supplémentaire** où l'oracle est
polynomial. Une preuve doit y survivre aussi. Le théorème d'Aaronson–Wigderson est
**cité, non démontré** ; ce jouet en montre la raison d'être, et il ne prétend
rien sur `P ≠ NP`.

## 4. Bornes inférieures de circuits : du comptage à l'explicite

Trois paliers, du plus faible au plus concret :

1. **le comptage** (Shannon) — combien de fonctions sont *au plus* atteignables
   avec `s` portes ? On compare le **décompte théorique** des configurations de
   circuits à la **clôture exhaustive** des fonctions réellement atteignables ;
2. **l'objet explicite** — le plan de Fano et son graphe d'incidence de Heawood :
   14 sommets, 21 arêtes, 3-régulier, sans 4-cycle ;
3. **la borne du registre** — une forme restrictive (forme **étoile**) dont le
   protocole public énonce une formule en `|E|` et `τ` : `|E| + τ − 1` portes.
   Cette formule est **attribuée** au registre du site archivé, avec sa
   restriction et sa date de relevé ; elle est **ici recalculée** sur des
   graphes où le calcul est entier et vérifiable.

Le graphe est ici un **graphe simple** : `E` est l'ensemble des arêtes, `τ` la
taille d'un couplage maximum (matching maximum).


In [7]:
// --- Comptage (Shannon) et cloture exhaustive ------------------------------
/// Decompte (majorant) des configurations de circuits de s portes.
/// Portes AND/OR (fan-in 2) et NOT (fan-in 1), signaux disponibles : les n
/// entrees et les s sorties de portes. Chaque porte : 3 types et au plus
/// (n+s)^2 choix de fils. Majorant volontairement grossier.
long Portee(int n, int s) {
    long total = (long)Math.Pow(3, s);
    for (int i = 0; i < 2 * s; i++) total *= (n + s);
    return total;
}

/// Fonctions booleennes a n variables atteignables par <= k portes.
/// BFS sur les ensembles de fonctions (une fonction = un entier de 2^n bits).
(int[] parNiveau, int total) Cloture(int n, int k) {
    int mask = (1 << (1 << n)) - 1;
    var niveau = new HashSet<int>();
    for (int i = 0; i < n; i++) {
        int f = 0;
        for (int x = 0; x < (1 << n); x++) if (((x >> i) & 1) == 1) f |= 1 << x;
        niveau.Add(f);
    }
    var vus = new HashSet<int>(niveau);
    var parNiveau = new Dictionary<int, int> { [0] = vus.Count };
    var courant = new List<int>(niveau);
    for (int p = 1; p <= k; p++) {
        var nouveau = new HashSet<int>();
        foreach (int a in courant) {
            nouveau.Add((~a) & mask);
            foreach (int b in courant) { nouveau.Add(a & b); nouveau.Add(a | b); }
        }
        nouveau.ExceptWith(vus);
        vus.UnionWith(nouveau);
        parNiveau[p] = nouveau.Count;
        courant = nouveau.ToList();
        if (courant.Count == 0) break;
    }
    return (parNiveau.OrderBy(kv => kv.Key).Select(kv => kv.Value).ToArray(), vus.Count);
}

int n3 = 3;
Console.WriteLine($"n = {n3} variables -> {1 << (1 << n3)} fonctions booleennes au total");
var cl = Cloture(n3, 4);
int cumul = 0;
for (int p = 0; p < cl.parNiveau.Length; p++) {
    cumul += cl.parNiveau[p];
    Console.WriteLine($"  <= {p} porte(s) : +{cl.parNiveau[p],4} fonctions atteintes (cumul {cumul})");
}

Console.WriteLine();
Console.WriteLine("Comparaison avec le decompte theorique des configurations :");
for (int s = 1; s <= 4; s++)
    Console.WriteLine($"  s = {s} : configurations majorantes = {Portee(n3, s),10} (fonctions totales = {1 << (1 << n3)})");

// La borne de comptage : il faut que le decompte des configurations depasse
// le nombre de fonctions pour qu'aucune fonction ne soit forcee d'exiger s portes.
int s2 = 1;
while (Portee(n3, s2) < (1 << (1 << n3))) s2++;
Console.WriteLine();
Console.WriteLine($"borne de comptage (majorant grossier) : une fonction exige au moins {s2} portes");
Console.WriteLine($"cloture exhaustive mesuree : {cl.total} fonctions sur {1 << (1 << n3)} sont atteignables en <= 4 portes");


n = 3 variables -> 256 fonctions booleennes au total


  <= 0 porte(s) : +   3 fonctions atteintes (cumul 3)


  <= 1 porte(s) : +   9 fonctions atteintes (cumul 12)


  <= 2 porte(s) : +  40 fonctions atteintes (cumul 52)


  <= 3 porte(s) : + 114 fonctions atteintes (cumul 166)


  <= 4 porte(s) : +  88 fonctions atteintes (cumul 254)


... (3 outputs left out)

  s = 3 : configurations majorantes =    1259712 (fonctions totales = 256)


  s = 4 : configurations majorantes =  466948881 (fonctions totales = 256)


borne de comptage (majorant grossier) : une fonction exige au moins 2 portes


cloture exhaustive mesuree : 254 fonctions sur 256 sont atteignables en <= 4 portes


### Lecture du résultat — le décompte est un majorant, la clôture est un fait

Le décompte des configurations (`3^s (n+s)^{2s}`) est un **majorant** : il compte
des configurations de circuits, dont plusieurs décrivent la même fonction. La
clôture exhaustive, elle, compte des **fonctions** : elle est au plus égale, et
l'écart entre les deux est exactement la redondance (deux circuits, une fonction).

La borne « au moins `s` portes » sort du minorant `3^s (n+s)^{2s} ≥ 2^{2^n}` :
à `n = 3`, la clôture exhaustive montre que la quasi-totalité des 256 fonctions
est déjà atteinte en quatre portes — la borne de comptage est ici **très faible**,
et c'est normal : elle ne devient intéressante qu'asymptotiquement. Le carnet la
mesure quand même, pour fixer l'échelle exacte avant l'objet explicite.


In [8]:
// --- L'objet explicite : plan de Fano, graphe de Heawood --------------------
/// Points et droites du plan de Fano (F_2^3 prive de {0}).
/// Points : les 7 vecteurs non nuls. Droites : les triplets {a, b, a xor b}.
(List<int> points, List<int[]> droites) PlanDeFano() {
    var points = Enumerable.Range(1, 7).ToList();
    // La cle canonique « a-b-c » deduplique les droites ET les trie comme le
    // tuple d'entiers du jumeau Python (les points sont des chiffres 1..7).
    var vus = new Dictionary<string, int[]>();
    foreach (int a in points)
        foreach (int b in points)
            if (a < b) {
                int[] d = { a, b, a ^ b };
                Array.Sort(d);   // la droite est un ENSEMBLE de trois points
                vus[string.Join("-", d)] = d;
            }
    var droites = vus.Keys.OrderBy(k => k, StringComparer.Ordinal).Select(k => vus[k]).ToList();
    return (points, droites);
}

/// Graphe biparti incidence : sommets points + droites, arites = incidence.
(int nSommets, List<(int a, int b)> aretes) GrapheIncidence(List<int> points, List<int[]> droites) {
    var aretes = new List<(int, int)>();
    for (int i = 0; i < droites.Count; i++)
        foreach (int p in droites[i])
            aretes.Add((points.Count + i, points.IndexOf(p)));
    return (points.Count + droites.Count, aretes);
}

/// Vrai s'il existe un cycle de longueur 4 (deux sommets a >= 2 voisins communs).
bool AUn4Cycle(List<(int a, int b)> aretes, int nSommets) {
    var voisins = new List<HashSet<int>>();
    for (int v = 0; v < nSommets; v++) voisins.Add(new HashSet<int>());
    foreach (var (a, b) in aretes) { voisins[a].Add(b); voisins[b].Add(a); }
    for (int v = 0; v < nSommets; v++)
        for (int w = v + 1; w < nSommets; w++)
            if (voisins[v].Intersect(voisins[w]).Count() >= 2) return true;
    return false;
}

/// Taille d'un couplage maximum (recherche exhaustive par backtracking).
int CouplageMax(List<(int a, int b)> aretes, int nSommets) {
    int meilleur = 0;
    var utilise = new bool[nSommets];
    void Explorer(int i, int taille) {
        if (taille + (aretes.Count - i) <= meilleur) return;
        if (i == aretes.Count) { meilleur = Math.Max(meilleur, taille); return; }
        Explorer(i + 1, taille);
        var (a, b) = aretes[i];
        if (!utilise[a] && !utilise[b]) {
            utilise[a] = true; utilise[b] = true;
            Explorer(i + 1, taille + 1);
            utilise[a] = false; utilise[b] = false;
        }
    }
    Explorer(0, 0);
    return meilleur;
}

/// Rang d'une matrice sur GF(2) (elimination de Gauss, lignes comme bits).
int RangGF2(int[,] m) {
    int rows = m.GetLength(0), cols = m.GetLength(1);
    var a = new int[rows, cols];
    for (int i = 0; i < rows; i++)
        for (int j = 0; j < cols; j++) a[i, j] = m[i, j] % 2;
    int r = 0;
    for (int c = 0; c < cols; c++) {
        int piv = -1;
        for (int i = r; i < rows; i++) if (a[i, c] == 1) { piv = i; break; }
        if (piv < 0) continue;
        for (int j = 0; j < cols; j++) { int t = a[r, j]; a[r, j] = a[piv, j]; a[piv, j] = t; }
        for (int i = 0; i < rows; i++)
            if (i != r && a[i, c] == 1)
                for (int j = 0; j < cols; j++) a[i, j] ^= a[r, j];
        r++;
    }
    return r;
}

var (points, droites) = PlanDeFano();
var (nSommets, aretes) = GrapheIncidence(points, droites);
int E = aretes.Count;
int tau = CouplageMax(aretes, nSommets);

var degresGraphe = new Dictionary<int, int>();
foreach (var (a, b) in aretes) {
    degresGraphe[a] = degresGraphe.GetValueOrDefault(a) + 1;
    degresGraphe[b] = degresGraphe.GetValueOrDefault(b) + 1;
}

Console.WriteLine("Plan de Fano et graphe d'incidence de Heawood :");
Console.WriteLine($"  points            : {points.Count}");
Console.WriteLine($"  droites           : {droites.Count}");
Console.WriteLine($"  sommets           : {nSommets}");
Console.WriteLine($"  aretes |E|        : {E}");
Console.WriteLine($"  regularite        : [{string.Join(", ", degresGraphe.Values.Distinct().OrderBy(x => x))}]");
Console.WriteLine($"  4-cycle present   : {AUn4Cycle(aretes, nSommets)}");
Console.WriteLine($"  couplage max tau : {tau}");
Console.WriteLine($"  |E| + tau - 1     : {E} + {tau} - 1 = {E + tau - 1}");

// La matrice d'incidence 7x7 sur GF(2) : son rang est un fait mesure.
var M = new int[7, 7];
for (int i = 0; i < droites.Count; i++)
    foreach (int p in droites[i]) M[i, points.IndexOf(p)] = 1;
Console.WriteLine($"  rang de la matrice d'incidence sur GF(2) : {RangGF2(M)}");


Plan de Fano et graphe d'incidence de Heawood :


  points            : 7


  droites           : 7


  sommets           : 14


  aretes |E|        : 21


  regularite        : [3]


  4-cycle present   : False


  couplage max tau : 7


  |E| + tau - 1     : 21 + 7 - 1 = 27


  rang de la matrice d'incidence sur GF(2) : 4


### Lecture du résultat — l'objet entier, recalculé dans la cellule

Le plan de Fano est reconstruit **depuis zéro** (les vecteurs non nuls de
`GF(2)^3`, les droites `{a, b, a ⊕ b}`) et tout est **recalculé** : 14 sommets,
21 arêtes, graphe 3-régulier, **absence de 4-cycle** (test : deux sommets à deux
voisins communs), couplage maximum `τ = 7` par backtracking exhaustif. La
formule du registre donne alors `|E| + τ − 1 = 21 + 7 − 1 = 27`.

Le rang de la matrice d'incidence sur GF(2) (4) est un fait supplémentaire du
même objet — il relie la combinatoire du plan à l'algèbre linéaire sur GF(2), donc
aux deux sections précédentes.

**Ce qui n'est pas fait ici** : la conjecture du registre n'est **pas** vérifiée.
Ce que le carnet calcule, c'est la quantité combinatoire `|E| + τ − 1` sur un
objet qu'il construit ; le registre, lui, énonce une borne sur une famille de
circuits **en forme étoile**, restriction que l'on ne réinstalle pas ici. Les deux
n'ont pas le même statut : l'un est **calculé**, l'autre est **cité**.


In [9]:
// --- Temoin 4 (negatif) : un graphe AVEC 4-cycle ---------------------------
// K_{2,2} : 4 sommets, 4 aretes, un 4-cycle.
var aretesK = new List<(int, int)> { (0, 2), (0, 3), (1, 2), (1, 3) };
int tauK = CouplageMax(aretesK, 4);
Console.WriteLine("Temoin 4 (negatif) -- K_{2,2} :");
Console.WriteLine($"  |E| = {aretesK.Count}, tau = {tauK}, |E| + tau - 1 = {aretesK.Count + tauK - 1}");
Console.WriteLine($"  4-cycle present : {AUn4Cycle(aretesK, 4)}");
Console.WriteLine("  => l'hypothese « pas de 4-cycle » de la forme etoile EST VIOLEE :");
Console.WriteLine("     la formule ne s'applique pas, et le calcul ne pretend rien dire d'elle.");

// --- Temoin 5 : les deux petits cas releves dans l'archive -----------------
Console.WriteLine();
Console.WriteLine("Temoin 5 -- les deux petits cas, recalcules :");

// Chemin sur 4 sommets.
var aretesP4 = new List<(int, int)> { (0, 1), (1, 2), (2, 3) };
// Cycle sur 6 sommets.
var aretesC6 = new List<(int, int)>();
for (int i = 0; i < 6; i++) aretesC6.Add((i, (i + 1) % 6));

foreach (var (nom, ar, nbSommets) in new[] {
    ("P4 (chemin sur 4 sommets)", aretesP4, 4),
    ("C6 (cycle sur 6 sommets)", aretesC6, 6),
}) {
    int t = CouplageMax(ar, nbSommets);
    Console.WriteLine($"  {nom,-30} |E| = {ar.Count}, tau = {t}, |E| + tau - 1 = {ar.Count + t - 1}");
}


Temoin 4 (negatif) -- K_{2,2} :


  |E| = 4, tau = 2, |E| + tau - 1 = 5


  4-cycle present : True


  => l'hypothese « pas de 4-cycle » de la forme etoile EST VIOLEE :


     la formule ne s'applique pas, et le calcul ne pretend rien dire d'elle.


Temoin 5 -- les deux petits cas, recalcules :


  P4 (chemin sur 4 sommets)      |E| = 3, tau = 2, |E| + tau - 1 = 4


  C6 (cycle sur 6 sommets)       |E| = 6, tau = 3, |E| + tau - 1 = 8


### Lecture du résultat — un témoin négatif, deux petits cas provisoires

Le témoin négatif est **décisif** pour la discipline du carnet : sur `K_{2,2}`, la
formule donnerait `4 + 2 − 1 = 5`, mais l'hypothèse « pas de 4-cycle » est
violée. Une formule qui s'applique partout n'est pas une borne ; une formule dont
**l'hypothèse** peut échouer doit être citée avec elle.

Les deux petits cas (`P4` et `C6`) sont **recalculés** et **marqués provisoires** :
ils proviennent du relevé archivé de l'issue, dont l'énoncé exact a été **retiré
avec remplacement** par le registre lui-même (« same term »). Le carnet les
recalcule donc comme un **contrôle de la formule**, pas comme un contrôle de
l'énoncé du registre.

**Rappel de protocole** : le registre réécrit ses énoncés. Toute borne citée
depuis ce flux doit être relue dans sa forme exacte, à la date de son relevé.


## 5. La frontière vivante : lire le registre sans le reprendre

Le protocole du registre, lu dans l'archive :

- une **conjecture** est proposée par un siège, avec un énoncé textuel ;
- elle est **vérifiée** par un juge extérieur (« the checker accepted … on an
  outside runner ») ;
- elle est **réfutée** par la vérification de sa **négation** (« its negation,
  L-meta-(N−1), is checked ») ;
- elle est parfois **retirée avec remplacement et motif** (`lemma_retired: … replaced
  by …: … plain now says … ; same term`) ;
- le flux porte **le texte** de l'énoncé, et le registre **le réécrit**.

La cellule ci-dessous relit la **note de doctrine versionnée dans le dépôt**
(`docs/reference/registres-de-preuves-externes.md`), qui cite le flux Atom archivé
et son relevé d'issue : elle compte les catégories, restitue les six règles du
protocole avec leur fichier source, et **vérifie** que la borne calculée dans ce
carnet n'est pas présentée comme la leur. Les deux objets sont printés côte à
côte, chacun avec sa source.

La donnée lue est **dans le dépôt**, pas sur le poste : la sortie du carnet ne
dépend donc d'aucune machine, et l'archive complète du registre (avec son
manifeste et ses empreintes) reste au gisement partagé, sans jamais être
commitée.

In [10]:
// --- Lecture de la note de doctrine versionnee -------------------------------
// La source lue est la NOTE DE DOCTRINE versionnee dans le depot
// (`docs/reference/registres-de-preuves-externes.md`), et NON l'archive du
// gisement partage : la note est presente sur toute machine, donc la sortie de
// ce carnet ne depend pas du poste. L'archive complete du registre (flux Atom,
// manifeste, empreintes) reste au gisement partage et n'est jamais committee --
// la note en cite les formes, elle ne recopie pas la publication.
string relative = Path.Combine("docs", "reference", "registres-de-preuves-externes.md");
string note = "";
for (string d = Directory.GetCurrentDirectory(); !string.IsNullOrEmpty(d); d = Path.GetDirectoryName(d)) {
    string candidat = Path.Combine(d, relative);
    if (File.Exists(candidat)) { note = candidat; break; }
}
if (note == "") throw new Exception("NOTE ABSENTE : " + relative + " introuvable au-dessus de " + Directory.GetCurrentDirectory());

// La forme relevee est entre guillemets francais : on les retire, puis les
// espaces residuels, pour que la ligne citee ne porte pas de double espace.
string Nettoie(string s) => s.Trim().Trim('«', '»').Trim();

string texte = File.ReadAllText(note);
Console.WriteLine("Note de doctrine (versionnee dans le depot) : " + Path.GetFileName(note));

// Le comptage de la fenetre relevee, lu dans la note.
var fenetre = Regex.Match(texte, @"contient (\d+) entr[^:]*: (\d+) `lemma_proposed`, (\d+) `lemma_checked`, (\d+) `lemma_refuted`");
if (!fenetre.Success) throw new Exception("Le comptage de la fenetre relevee est introuvable dans la note.");
Console.WriteLine($"Fenetre relevee : {fenetre.Groups[1].Value} entrees -> {fenetre.Groups[2].Value} lemma_proposed, {fenetre.Groups[3].Value} lemma_checked, {fenetre.Groups[4].Value} lemma_refuted");
Console.WriteLine();

// Les six regles du protocole : forme relevee et fichier source, a cote.
Console.WriteLine("Regles du protocole relevees dans la note (source a cote) :");
foreach (string ligne in texte.Split('\n').Where(l => Regex.IsMatch(l, @"^\| R\d \|"))) {
    var champs = ligne.Split('|').Select(c => c.Trim()).Where(c => c.Length > 0).ToList();
    if (champs.Count < 4) continue;
    Console.WriteLine($"  {champs[0]}  {champs[1]}");
    Console.WriteLine($"        « {Nettoie(champs[2])} »   [{champs[3].Trim('`')}]");
}
Console.WriteLine();

// Les entrees de la fenetre qui parlent du plan de Fano : l'objet de la section 4.
Console.WriteLine("Entrees mentionnant le plan de Fano (verbatim, releve de la note) :");
foreach (string ligne in texte.Split('\n').Where(l => l.Contains("Fano plane:"))) {
    foreach (Match g in Regex.Matches(ligne, "«([^»]*)»")) Console.WriteLine("  - " + Nettoie(g.Groups[1].Value));
}
Console.WriteLine();

Console.WriteLine("NOTRE calcul (section 4)  : |E| + tau - 1 = 27 sur le graphe de Heawood reconstruit ici");
Console.WriteLine("LEUR enonce (registre)    : une borne en forme etoile sur une famille de circuits");
Console.WriteLine("=> deux enonces distincts ; le registre n'est ni verifie ni repris par ce carnet.");

Note de doctrine (versionnee dans le depot) : registres-de-preuves-externes.md


Fenetre relevee : 30 entrees -> 13 lemma_proposed, 13 lemma_checked, 4 lemma_refuted


Regles du protocole relevees dans la note (source a cote) :


  R1  Le noyau revérifie, sans axiome hors des trois standards


        « The checker builds the file in a pinned environment, has the kernel verify it a second time, and accepts it only if it proves exactly the recorded statement and rests on no gap and no axiom outside the standard three. »   [how.html]


  R2  Le juge ne juge pas la **fidélité** de l'énoncé


... (11 outputs left out)

  - Conjecture: seat-intake-1 proposed L-meta-11461: A step toward: Fano plane: a circuit of at most 27 AND/OR gates of fan-in two (distinct earlier wires) computes incidence and has a double… (step 1 of 2)


NOTRE calcul (section 4)  : |E| + tau - 1 = 27 sur le graphe de Heawood reconstruit ici


LEUR enonce (registre)    : une borne en forme etoile sur une famille de circuits


=> deux enonces distincts ; le registre n'est ni verifie ni repris par ce carnet.


### Lecture du résultat — deux énoncés distincts, aucune confusion

La cellule affiche côte à côte **notre** calcul (`|E| + τ − 1 = 27`, sur un graphe
construit dans le carnet) et **leur** énoncé (une borne sur des circuits en forme
étoile, dans un registre dont le protocole fait autorité sur lui-même et sur
personne d'autre).

Les deux énoncés sont lus dans la **note versionnée dans le dépôt** : le nôtre
vient du graphe reconstruit ici, le leur du flux archivé que la note cite avec
son fichier source et son empreinte. Les deux « 27 » ne se confondent pas — ils
ont des **sens opposés** (l'un annonce un circuit, l'autre en borne la taille).

Ce que le registre **ne** nous autorise pas à faire : présenter l'une de ses
conjectures comme un résultat acquis, ou son absence de réfutation comme une
démonstration. Ce qu'il **nous** donne : un cas d'étude d'épistémologie en direct,
où la forme de l'énoncé change, où une réfutation est une **vérification de la
négation**, et où un retrait est motivé.

## 6. Exercices

Trois exercices, sur le modèle de la série — **stubs non bloquants** (C.1) : le
carnet s'exécute de bout en bout même les exercices non complétés.


In [11]:
/// Exercice 1. Degré de l'extension multilineaire sur GF(p) d'une table.
/// Rappel : l'extension multilineaire de f sur GF(p) a pour coefficients
/// c_S = somme_{T sous-ensemble de S} de (-1)^{|S|-|T|} f(1_T) (mod p) ; le
/// degré est le plus grand |S| tel que c_S != 0.
/// TODO etudiant : implementer l'inversion de Moebius sur Z/p et retourner le
/// degré. Les fonctions utilitaires du carnet sont dans la section 3.
int? DegréDunePropriété(int[] f, int p = 2) {
    int? result = null; // TODO etudiant
    return result;
}

Console.WriteLine("Exercice 1 : degre_dune_propriete renvoie " + (DegréDunePropriété(new[] { 0, 1, 1, 0 })?.ToString() ?? "null"));


Exercice 1 : degre_dune_propriete renvoie null


**Exercice 2** — la borne en forme étoile est une **formule sous hypothèse**.
Reprendre le graphe de Heawood de la section 4, et vérifier par vous-même que
la formule `|E| + τ − 1` **dépend** de l'hypothèse « pas de 4-cycle » :
construire un graphe où elle échoue, et écrire la phrase qui rendrait la citation
honnête.


In [12]:
/// Exercice 2. Calcule |E| + tau - 1 et teste l'hypothese « pas de 4-cycle ».
/// TODO etudiant : renvoyer un dictionnaire avec les cles "E", "tau", "borne" et
/// "hypothese_tient" (False si un 4-cycle est present). Le carnet fournit
/// CouplageMax et AUn4Cycle.
Dictionary<string, object>? BorneEtoile(List<(int a, int b)> aretes, int nSommets) {
    Dictionary<string, object>? result = null; // TODO etudiant
    return result;
}

var essai = BorneEtoile(new List<(int, int)> { (0, 2), (0, 3), (1, 2), (1, 3) }, 4);
Console.WriteLine("Exercice 2 : borne_etoile renvoie " + (essai is null ? "null" : string.Join(", ", essai.Select(kv => kv.Key + "=" + kv.Value))));


Exercice 2 : borne_etoile renvoie null



(5,27): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(6,31): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



**Exercice 3** — le mur du théorème de Razborov–Rudich est une colonne, pas une
mesure. Reprendre le tableau des densités de la section 2 et ajouter une
**quatrième colonne** : « la propriété est-elle *utile* ? ». Dire précisément ce
qu'un tiers devrait fournir pour y répondre, et pourquoi la mesure du carnet ne
peut pas y répondre.


In [13]:
/// Exercice 3. Colonne « utile ? » du tableau des propriétés.
/// TODO etudiant : renvoyer, pour chaque propriété, une phrase qui dit soit
/// « non : la propriété est une tautologie ou sa négation l'est », soit « le
/// mur : troisième colonne non atteinte, le théorème est cité et non re-prouvé ».
List<string>? ColonnesUtilité(Dictionary<string, double> mesurees) {
    List<string>? result = null; // TODO etudiant
    return result;
}

Console.WriteLine("Exercice 3 : colonnes_utilite renvoie " + (ColonnesUtilité(mesurees) is null ? "null" : "une phrase par propriete"));


Exercice 3 : colonnes_utilite renvoie null



(5,13): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(6,17): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



## 7. Conclusion

| Geste | Où | Ce qui a été fait | Niveau |
|---|---|---|---|
| Trois barrières attribuées | §1 | Relativisation (rappelée, traitée en 03/03c), natural proofs, algébrisation | **cité** |
| Densité et coût de calcul | §2 | Population **complète** des 65 536 fonctions à 4 variables : parité, degrés, sensibilité ; famille pseudo-aléatoire (K = 256) | **mesuré** |
| Degré d'extension multilinéaire | §3 | AND, OR, PARITY, MAJORITY : degrés sur GF(2), GF(3), GF(5) et nombre de portes d'un circuit **construit** | **mesuré** |
| Jouet d'oracles | §3 | Deux mondes de même densité et de même degré, séparés par un test algébrique | **mesuré** |
| Comptage et clôture exhaustive | §4 | `n = 3`, portée des configurations contre fonctions atteintes en ≤ 4 portes | **mesuré** |
| Objet explicite | §4 | Plan de Fano, graphe de Heawood, `arêtes = 21`, `τ = 7`, `21 + 7 − 1 = 27`, rang 4 sur GF(2) | **mesuré** |
| Témoin négatif | §4 | `K_{2,2}` : l'hypothèse « pas de 4-cycle » est violée | **mesuré** |
| Deux petits cas | §4 | `P4` → 4, `C6` → 8 — **provisoires**, énoncé du registre relu dans l'archive | **recalculé** |
| Registre du site archivé | §5 | Fenêtre relue, formes de réfutation citées, énoncés non repris | **lu** |

## À retenir

1. **Trois barrières, trois régimes.** La relativisation demande que la preuve
   tienne dans deux mondes ; l'algébrisation en ajoute un, polynomial ; les natural
   proofs disent qu'un certain type de preuve **ne peut pas** exister sous hypothèse
   forte. La série traitait le premier ; ce carnet nomme les deux autres.
2. **Une borne se cite avec son hypothèse.** Sur `K_{2,2}`, la formule donne un
   nombre et l'hypothèse est violée : c'est le témoin négatif, pas un accident.
3. **Un registre qui réécrit ses énoncés** impose de dater ses citations. Le
   carnet lit l'archive datée, et ne présente rien de ce qu'elle contient comme un
   résultat acquis.
4. **Ce que ce carnet ne prouve pas** : ni `P ≠ NP`, ni la borne du registre, ni les
   théorèmes cités (Razborov–Rudich, Aaronson–Wigderson). Les hypothèses de ces
   théorèmes sont **attribuées**, jamais vérifiées ici.

## Vérifiez votre compréhension

1. Une propriété « grande » et « constructive » est-elle alors « utile » ? Que
   faudrait-il construire pour rendre une propriété utile ?
2. Dans la section 3, pourquoi le degré d'une fonction booléenne peut-il changer
   quand on change de corps ? Que se passerait-il avec une fonction à valeurs dans
   `F_p` qui n'est pas booléenne ?
3. Le témoin négatif `K_{2,2}` : quelle phrase faut-il écrire dans une PR qui cite
   la borne du registre, pour que la citation soit honnête ?
4. Le flux dit « its negation, L-meta-(N−1), is checked ». En quoi une réfutation
   obtenue par la négation est-elle plus solide qu'un échec du vérificateur ?

## Pour aller plus loin

- **Carnets voisins** : [03 — Plus de temps, plus de problèmes](Complexity-03-HierarchieTemporelle-Python.ipynb),
  [03c — Complexité Zoo](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb),
  [03d — L'ETH et sa réfutation annoncée](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb),
  [05 — Compter est plus dur que vérifier](Complexity-05-CountingHarder-Permanent-Python.ipynb),
  [05b — La permanente, frontière quantique](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb).
- **Le même objet, ailleurs dans le dépôt** : le graphe de Heawood est déjà l'objet
  de l'exercice 3 de
  [Frontieres-01 — Edge coloring et Tutte](../../Search/Part5-Frontieres/Frontieres-01-EdgeColoring-Tutte-Python.ipynb) ;
  le couplage hongrois est implémenté par
  [Sudoku-15](../../Sudoku/Sudoku-15-Infer-CSharp.ipynb) et la permanente
  est l'objet de `05`/`05b` — **le même objet combinatoire, trois régimes : compter,
  optimiser, borner**.
- **Sources**, toutes archivées dans le gisement partagé
  (`Bibliothèque IA/Automata/Complexity Theory and Hardness/`) :
  - Baker–Gill–Solovay, *Relativizations of the P versus NP Question*,
    SIAM J. Comput. 4(4), 1975 ;
  - Razborov–Rudich, *Natural Proofs*, STOC '94 ; version journal
    **J. Comput. Syst. Sci. 55(1):24–35, 1997** (copie archivée : manuscrit de
    l'auteur daté du 30-11-1996) ;
  - Aaronson–Wigderson, *Algebrization: A New Barrier in Complexity Theory*,
    STOC 2008 ; version étendue **ACM Transactions on Computation Theory 1(1),
    2009** — dite *TOCT* dans le tableau de la section 1 — (copie archivée :
    résumé étendu STOC, 10 pages).


In [14]:
// --- Controle de parite avec le jumeau Python (PR #28573) -------------------
// Les valeurs attendues sont celles des SORTIES COMMITTEES du jumeau Python.
// Elles sont reproductibles : la population est complete et la famille
// pseudo-aleatoire derive de SHA-256 (donc identique bit a bit dans les deux
// langages). Ce qui n'est PAS reproductible -- chronometres et tirage du monde
// B du jouet d'oracles -- n'est pas compare ici.
void Verifie(bool condition, string message) {
    if (!condition) throw new Exception("PARITE C# / Python rompue : " + message);
}

// --- Section 2 : population complete et densites ---------------------------
Verifie(tables.Length == 65536, "population : 65536 fonctions attendues, " + tables.Length + " mesurees");
Verifie(degres.Max() == 4, "degre maximal : 4 attendu, " + degres.Max() + " mesure");
Verifie(degres.Count(d => d == n) == 32768, "32768 fonctions de degre n attendues, " + degres.Count(d => d == n) + " mesurees");
Verifie(Math.Abs(mesurees["parite (deg == n)"] - 0.5000) <= 5e-5, "densite parite : 0.5000 attendu, " + mesurees["parite (deg == n)"] + " mesure");
Verifie(Math.Abs(mesurees["deg >= 1"] - 1.0000) <= 5e-5, "densite deg>=1 : 1.0000 attendu, " + mesurees["deg >= 1"] + " mesure");
Verifie(Math.Abs(mesurees["deg >= 2"] - 0.9995) <= 5e-5, "densite deg>=2 : 0.9995 attendu, " + mesurees["deg >= 2"] + " mesure");
Verifie(Math.Abs(mesurees["deg >= 3"] - 0.9688) <= 5e-5, "densite deg>=3 : 0.9688 attendu, " + mesurees["deg >= 3"] + " mesure");
Verifie(Math.Abs(mesurees["deg >= 4"] - 0.5000) <= 5e-5, "densite deg>=4 : 0.5000 attendu, " + mesurees["deg >= 4"] + " mesure");
Verifie(Math.Abs(mesurees["sensibilite moyenne >= 2"] - 0.9985) <= 5e-5, "densite sens>=2 : 0.9985 attendu, " + mesurees["sensibilite moyenne >= 2"] + " mesure");
Verifie(Math.Abs(mesurees["sensibilite moyenne >= 4"] - 0.6566) <= 5e-5, "densite sens>=4 : 0.6566 attendu, " + mesurees["sensibilite moyenne >= 4"] + " mesure");
Verifie(Math.Abs(mesurees["sensibilite moyenne >= 6"] - 0.0080) <= 5e-5, "densite sens>=6 : 0.0080 attendu, " + mesurees["sensibilite moyenne >= 6"] + " mesure");

// --- Section 2 : la famille pseudo-aleatoire (SHA-256, identique part a part)
Verifie(Math.Abs(mesPseudo["parite (deg == n)"] - 0.5195) <= 5e-5, "pseudo parite : 0.5195 attendu, " + mesPseudo["parite (deg == n)"] + " mesure");
Verifie(Math.Abs(mesPseudo["deg >= 3"] - 0.9805) <= 5e-5, "pseudo deg>=3 : 0.9805 attendu, " + mesPseudo["deg >= 3"] + " mesure");
Verifie(Math.Abs(mesPseudo["sensibilite moyenne >= 6"] - 0.0039) <= 5e-5, "pseudo sens>=6 : 0.0039 attendu, " + mesPseudo["sensibilite moyenne >= 6"] + " mesure");

// --- Section 2 : la loi de reference (200 tirages SplitMix64, identiques
// dans les deux langages) ---------------------------------------------
Verifie(Math.Abs(ecartTypeRef["parite (deg == n)"] - 1.03) <= 1e-2, "ecart-type de la loi (parite) : 1.03 attendu, " + ecartTypeRef["parite (deg == n)"] + " mesure");
Verifie(Math.Abs(ecartTypeRef["deg >= 3"] - 1.01) <= 1e-2, "ecart-type de la loi (deg>=3) : 1.01 attendu, " + ecartTypeRef["deg >= 3"] + " mesure");
Verifie(Math.Abs(ecartTypeRef["sensibilite moyenne >= 6"] - 1.02) <= 1e-2, "ecart-type de la loi (sens>=6) : 1.02 attendu, " + ecartTypeRef["sensibilite moyenne >= 6"] + " mesure");
Verifie(Math.Abs(partRef["parite (deg == n)"] - 0.58) <= 1e-9, "part des tirages aleatoires (parite) : 58,0 % attendue, " + (100 * partRef["parite (deg == n)"]).ToString("F1") + " % mesuree");
Verifie(Math.Abs(partRef["deg >= 3"] - 0.38) <= 1e-9, "part des tirages aleatoires (deg>=3) : 38,0 % attendue, " + (100 * partRef["deg >= 3"]).ToString("F1") + " % mesuree");
Verifie(Math.Abs(partRef["sensibilite moyenne >= 6"] - 0.48) <= 1e-9, "part des tirages aleatoires (sens>=6) : 48,0 % attendue, " + (100 * partRef["sensibilite moyenne >= 6"]).ToString("F1") + " % mesuree");

// --- Section 3 : degres par corps et nombre de portes -----------------------
Verifie(DegreeExtension(refs["AND"], 2) == 7, "degre AND sur GF(2) : 7 attendu");
Verifie(DegreeExtension(refs["AND"], 3) == 7, "degre AND sur GF(3) : 7 attendu");
Verifie(DegreeExtension(refs["AND"], 5) == 7, "degre AND sur GF(5) : 7 attendu");
Verifie(nAnd == 6, "portes AND : 6 attendues, " + nAnd + " mesurees");
Verifie(DegreeExtension(refs["OR"], 2) == 7 && DegreeExtension(refs["OR"], 3) == 7 && DegreeExtension(refs["OR"], 5) == 7, "degres OR : 7/7/7 attendus");
Verifie(nOr == 6, "portes OR : 6 attendues, " + nOr + " mesurees");
Verifie(DegreeExtension(refs["PARITY"], 2) == 1, "degre PARITY sur GF(2) : 1 attendu");
Verifie(DegreeExtension(refs["PARITY"], 3) == 7, "degre PARITY sur GF(3) : 7 attendu");
Verifie(DegreeExtension(refs["PARITY"], 5) == 7, "degre PARITY sur GF(5) : 7 attendu");
Verifie(nPar == 30, "portes PARITY : 30 attendues, " + nPar + " mesurees");
Verifie(DegreeExtension(refs["MAJORITY"], 2) == 4, "degre MAJORITY sur GF(2) : 4 attendu");
Verifie(DegreeExtension(refs["MAJORITY"], 3) == 7, "degre MAJORITY sur GF(3) : 7 attendu");
Verifie(DegreeExtension(refs["MAJORITY"], 5) == 5, "degre MAJORITY sur GF(5) : 5 attendu");
Verifie(nMaj == 139, "portes MAJORITY : 139 attendues, " + nMaj + " mesurees");
Verifie(nOkMaj == (1 << n7), "controle du circuit MAJORITY : " + (1 << n7) + " affectations attendues, " + nOkMaj + " mesurees");
Verifie(nAnd2 == 1, "portes AND_2 : 1 attendue, " + nAnd2 + " mesuree");
Verifie(DegreeExtension(new[] { 0, 0, 0, 1 }, 2) == 2, "degre AND_2 sur GF(2) : 2 attendu");
Verifie(DegreeExtension(f, 2) == 3, "controle z1 z2 z3 : degre 3 attendu");

// --- Section 3 : le monde A du jouet d'oracles (deterministe) --------------
Verifie(tripletsFano == 7, "triplets nuls du plan de Fano : 7 attendus, " + tripletsFano + " mesures");
Verifie(degA == 4, "degre de l'indicateur du monde A : 4 attendu, " + degA + " mesure");

// --- Section 4 : comptage et cloture exhaustive ----------------------------
Verifie(cl.parNiveau[0] == 3, "cloture niveau 0 : 3 fonctions attendues, " + cl.parNiveau[0] + " mesurees");
Verifie(cl.parNiveau[1] == 9, "cloture niveau 1 : 9 fonctions attendues, " + cl.parNiveau[1] + " mesurees");
Verifie(cl.parNiveau[2] == 40, "cloture niveau 2 : 40 fonctions attendues, " + cl.parNiveau[2] + " mesurees");
Verifie(cl.parNiveau[3] == 114, "cloture niveau 3 : 114 fonctions attendues, " + cl.parNiveau[3] + " mesurees");
Verifie(cl.parNiveau[4] == 88, "cloture niveau 4 : 88 fonctions attendues, " + cl.parNiveau[4] + " mesurees");
Verifie(cl.total == 254, "cloture totale : 254 fonctions attendues, " + cl.total + " mesurees");
Verifie(s2 == 2, "borne de comptage : 2 portes attendues, " + s2 + " mesurees");

// --- Section 4 : l'objet explicite ------------------------------------------
Verifie(points.Count == 7 && droites.Count == 7, "plan de Fano : 7 points et 7 droites attendus");
Verifie(nSommets == 14, "sommet(s) du graphe d'incidence : 14 attendus, " + nSommets + " mesures");
Verifie(E == 21, "aretes |E| : 21 attendues, " + E + " mesurees");
Verifie(!AUn4Cycle(aretes, nSommets), "le graphe de Heawood ne doit pas avoir de 4-cycle");
Verifie(tau == 7, "couplage maximum : 7 attendu, " + tau + " mesure");
Verifie(E + tau - 1 == 27, "borne |E| + tau - 1 : 27 attendue, " + (E + tau - 1) + " mesuree");
Verifie(RangGF2(M) == 4, "rang de la matrice d'incidence sur GF(2) : 4 attendu");

// --- Section 4 : temoin negatif et petits cas -------------------------------
Verifie(aretesK.Count == 4 && tauK == 2, "K_{2,2} : |E| = 4 et tau = 2 attendus");
Verifie(AUn4Cycle(aretesK, 4), "K_{2,2} doit contenir un 4-cycle");
Verifie(aretesK.Count + tauK - 1 == 5, "K_{2,2} : la formule donne 5");
Verifie(CouplageMax(aretesP4, 4) == 2 && aretesP4.Count + CouplageMax(aretesP4, 4) - 1 == 4, "P4 : |E| = 3, tau = 2, borne 4 attendus");
Verifie(CouplageMax(aretesC6, 6) == 3 && aretesC6.Count + CouplageMax(aretesC6, 6) - 1 == 8, "C6 : |E| = 6, tau = 3, borne 8 attendus");

Console.WriteLine("PARITE C# / Python : toutes les grandeurs reproductibles coincident.");
Console.WriteLine();
Console.WriteLine("PARITE C# / Python : toutes les grandeurs reproductibles coincident.");
Console.WriteLine();
Console.WriteLine($"Le seuil MAJORITY (n = 7) : {nMaj} portes des deux cotes, et le calcul de la fonction est");
Console.WriteLine($"controle sur les {1 << n7} affectations ({nOkMaj} / {1 << n7} des deux cotes).");
Console.WriteLine("La loi de reference des ecarts est comparee aussi : 200 tirages SplitMix64, identiques");
Console.WriteLine("bit a bit dans les deux langages.");


PARITE C# / Python : toutes les grandeurs reproductibles coincident.


PARITE C# / Python : toutes les grandeurs reproductibles coincident.


Le seuil MAJORITY (n = 7) : 139 portes des deux cotes, et le calcul de la fonction est


controle sur les 128 affectations (128 / 128 des deux cotes).


La loi de reference des ecarts est comparee aussi : 200 tirages SplitMix64, identiques


bit a bit dans les deux langages.
